In [ ]:


!pip install gradio scikit-learn -q

import gradio as gr
import pandas as pd
import numpy as np
import random
import json
from datetime import datetime, timedelta
from sklearn.linear_model import SGDClassifier
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings("ignore")



OUTCOME_REWARD = {
    "converted":  1.0,   # Clicked + purchased
    "clicked":    0.7,   # Clicked notification
    "ignored":   -0.3,   # Delivered, not opened
    "opted_out": -1.0,   # Unsubscribed
}


customer_history = {}   # { cust_id: [ {score, outcome, reward, timestamp}, ... ] }


customer_threshold_delta = {}  # { cust_id: float }


ml_model   = SGDClassifier(loss="log_loss", learning_rate="optimal",
                            max_iter=1, warm_start=True, random_state=42)
scaler     = StandardScaler()
ml_trained = False   # flips True after first training event
feedback_log = []    # running log of all feedback events



def compute_receptivity_score(row):
    score = 50
    breakdown = {}

    if row["do_not_disturb"]:
        score -= 30; breakdown["do_not_disturb"] = -30
    if row["phone_face_down"]:
        score -= 15; breakdown["phone_face_down"] = -15
    if row["recent_error_encountered"]:
        score -= 10; breakdown["recent_friction"] = -10
    if row["recently_closed_app"]:
        score -= 10; breakdown["recently_closed_app"] = -10

    hour = row["hour_of_day"]
    if 8 <= hour <= 11:
        score += 20; breakdown["time_morning"] = +20
    elif 19 <= hour <= 21:
        score += 10; breakdown["time_evening"] = +10
    elif 0 <= hour <= 5:
        score -= 20; breakdown["time_latenight"] = -20
    elif row.get("is_rush_hour", False):
        score -= 10; breakdown["time_rushhour"] = -10

    if row.get("is_weekend", False):
        score -= 5; breakdown["weekend_penalty"] = -5

    brightness = row["screen_brightness"]
    if brightness < 15:
        score -= 15; breakdown["screen_dark"] = -15
    elif brightness > 70:
        score += 10; breakdown["screen_bright"] = +10

    if row.get("on_wifi", False):
        score += 8; breakdown["on_wifi"] = +8

    if row["swipe_speed"] == "slow":
        score += 15; breakdown["swipe_slow"] = +15
    elif row["swipe_speed"] == "fast":
        score -= 10; breakdown["swipe_fast"] = -10

    if row["session_depth"] == "deep":
        score -= 15; breakdown["session_deep"] = -15
    elif row["session_depth"] == "shallow":
        score += 10; breakdown["session_shallow"] = +10

    if row["time_on_current_screen_sec"] > 120:
        score += 8; breakdown["long_dwell"] = +8
    elif row["time_on_current_screen_sec"] < 10:
        score -= 8; breakdown["short_dwell"] = -8

    if row["sentiment_score"] > 0.5:
        score += 15; breakdown["positive_sentiment"] = +15
    elif row["sentiment_score"] < -0.5:
        score -= 15; breakdown["negative_sentiment"] = -15

    if row["avg_response_time_sec"] < 5:
        score -= 5; breakdown["response_fast"] = -5
    elif 5 <= row["avg_response_time_sec"] <= 15:
        score += 5; breakdown["response_calm"] = +5

    return max(0, min(100, score)), breakdown




def get_effective_threshold(customer_id):
    """
    Base send threshold = 50.
    Adjusted UP if customer historically ignores at this score.
    Adjusted DOWN if customer consistently engages.
    Capped between -25 and +25.
    """
    delta = customer_threshold_delta.get(customer_id, 0.0)
    return 50 + delta


def update_threshold(customer_id, outcome):
    """
    Online update: shift threshold based on outcome.
    Ignored/opted-out → raise the bar (be more conservative)
    Clicked/converted → lower the bar slightly (customer is receptive)
    """
    reward = OUTCOME_REWARD[outcome]
    current = customer_threshold_delta.get(customer_id, 0.0)

    if reward < 0:
        # Bad outcome → raise threshold (be more selective next time)
        shift = abs(reward) * 5.0
        new_delta = min(current + shift, 25.0)
    else:
        # Good outcome → slightly lower threshold (customer is open)
        shift = reward * 2.0
        new_delta = max(current - shift, -25.0)

    customer_threshold_delta[customer_id] = round(new_delta, 2)
    return new_delta


def row_to_features(row):
    """Convert signal row to flat feature vector for ML model."""
    return np.array([
        row["hour_of_day"] / 23.0,
        row["screen_brightness"] / 100.0,
        row["sentiment_score"],
        float(row["do_not_disturb"]),
        float(row["phone_face_down"]),
        float(row.get("on_wifi", False)),
        float(row["recent_error_encountered"]),
        float(row["recently_closed_app"]),
        ["slow","medium","fast"].index(row["swipe_speed"]) / 2.0,
        ["shallow","mid","deep"].index(row["session_depth"]) / 2.0,
        row["time_on_current_screen_sec"] / 300.0,
        row["avg_response_time_sec"] / 30.0,
    ]).reshape(1, -1)


def ml_update(row, outcome):
    """Online learning: update model with one new feedback event."""
    global ml_trained, scaler

    reward  = OUTCOME_REWARD[outcome]
    label   = 1 if reward > 0 else 0   # 1=receptive, 0=not receptive
    X       = row_to_features(row)

    if not ml_trained:
        # First event — initialize scaler
        scaler.fit(X)
        ml_trained = True

    X_scaled = scaler.transform(X)
    ml_model.partial_fit(X_scaled, [label], classes=[0, 1])


def ml_confidence(row):
    """Get ML model's confidence that customer is receptive (0–1)."""
    if not ml_trained:
        return None
    X        = row_to_features(row)
    X_scaled = scaler.transform(X)
    prob     = ml_model.predict_proba(X_scaled)[0][1]  # P(receptive)
    return round(prob, 3)



def hybrid_decide(customer_id, rule_score, ml_conf, sentiment):
    """
    Combines:
    1. Rule-based receptivity score
    2. Per-customer learned threshold
    3. ML model confidence (when available)
    """
    threshold = get_effective_threshold(customer_id)

    # Blend ML confidence into score if model is trained
    if ml_conf is not None:
        # ML nudges score by up to ±15 points
        ml_nudge   = (ml_conf - 0.5) * 30
        final_score = round(rule_score + ml_nudge)
        source      = f"Rule({rule_score}) + ML nudge({ml_nudge:+.1f})"
    else:
        final_score = rule_score
        source      = f"Rule only (ML not trained yet)"

    final_score = max(0, min(100, final_score))

    MESSAGES = {
        "promo":   "🎁 Exclusive offer — 20% off your next order!",
        "nudge":   "👀 Your cart is waiting.",
        "restock": "✅ An item you liked is back in stock.",
    }
    msg = "promo" if sentiment > 0.4 else ("nudge" if sentiment < -0.3 else "restock")

    if final_score >= threshold + 30:
        action, channel = "✅ SEND NOW",      "Full Push"
    elif final_score >= threshold:
        action, channel = "🔕 SEND SILENT",   "Silent Badge"
    elif final_score >= threshold - 20:
        retry = (datetime.now()+timedelta(minutes=15)).strftime("%H:%M")
        action, channel = "⏳ HOLD — RETRY",  f"Scheduled @ {retry}"
    else:
        retry = (datetime.now()+timedelta(minutes=60)).strftime("%H:%M")
        action, channel = "🚫 DO NOT DISTURB","Suppressed"

    return final_score, action, channel, MESSAGES[msg], source, threshold




def build_row(hour, brightness, sentiment, dnd, face_down,
              on_wifi, recent_err, closed_app, swipe, depth, dwell, response_t):
    return {
        "hour_of_day": int(hour),
        "screen_brightness": int(brightness),
        "sentiment_score": float(sentiment),
        "do_not_disturb": bool(dnd),
        "phone_face_down": bool(face_down),
        "on_wifi": bool(on_wifi),
        "recent_error_encountered": bool(recent_err),
        "recently_closed_app": bool(closed_app),
        "swipe_speed": swipe,
        "session_depth": depth,
        "time_on_current_screen_sec": int(dwell),
        "avg_response_time_sec": int(response_t),
        "is_weekend": datetime.now().weekday() >= 5,
        "is_rush_hour": int(hour) in [8, 9, 17, 18, 19],
    }


def score_and_decide(customer_id, hour, brightness, sentiment, dnd, face_down,
                     on_wifi, recent_err, closed_app, swipe, depth, dwell, response_t):

    if not customer_id.strip():
        return "⚠️ Please enter a Customer ID first."

    row        = build_row(hour, brightness, sentiment, dnd, face_down,
                           on_wifi, recent_err, closed_app, swipe, depth, dwell, response_t)
    rule_score, breakdown = compute_receptivity_score(row)
    ml_conf    = ml_confidence(row)
    final_score, action, channel, message, source, threshold = \
        hybrid_decide(customer_id, rule_score, ml_conf, float(sentiment))

    # Store in customer history (pending outcome)
    if customer_id not in customer_history:
        customer_history[customer_id] = []
    customer_history[customer_id].append({
        "row": row, "rule_score": rule_score,
        "final_score": final_score, "action": action,
        "timestamp": datetime.now().strftime("%H:%M:%S"), "outcome": None
    })

    # History summary
    hist      = customer_history[customer_id]
    hist_lines = ""
    if len(hist) > 1:
        hist_lines = "\n  📜 SEND HISTORY\n"
        for h in hist[-4:]:
            outcome_str = h["outcome"] if h["outcome"] else "⏳ awaiting feedback"
            hist_lines += f"     {h['timestamp']} | Score:{h['final_score']} | {h['action'][:12]} | {outcome_str}\n"

    # Top signals
    top3 = sorted(breakdown.items(), key=lambda x: abs(x[1]), reverse=True)[:3]
    sig  = "\n".join([f"  {'🟢' if v>0 else '🔴'} {k}: {'+' if v>0 else ''}{v}" for k,v in top3])

    bar = "█" * int(final_score/5) + "░" * (20 - int(final_score/5))

    delta = customer_threshold_delta.get(customer_id, 0.0)
    delta_str = f"+{delta}" if delta >= 0 else str(delta)

    output = f"""
╔══════════════════════════════════════════════╗
  ⚡ RECEPTIVITY AI — Hybrid Decision Output
╚══════════════════════════════════════════════╝

  👤 Customer     : {customer_id}
  🧮 Score Source : {source}

  📊 RECEPTIVITY SCORE
     [{bar}]  {final_score}/100

  🎯 Learned Threshold : {threshold}/100  (base 50 {delta_str} learned)
  🤖 ML Confidence     : {"Not trained yet — feed outcomes below" if ml_conf is None else f"{ml_conf*100:.1f}% receptive"}

  🎯 DECISION   {action}
  📡 CHANNEL    {channel}
  💬 MESSAGE    {message}

  📌 TOP SIGNALS
{sig}
{hist_lines}
  ⏱  {datetime.now().strftime("%H:%M:%S")}
"""
    return output


def record_outcome(customer_id, outcome):
    """User logs what happened after notification was sent."""
    if not customer_id.strip():
        return "⚠️ Enter a Customer ID."
    if customer_id not in customer_history or not customer_history[customer_id]:
        return "⚠️ No send history for this customer. Score them first."

    # Update last pending send
    last = None
    for entry in reversed(customer_history[customer_id]):
        if entry["outcome"] is None:
            last = entry
            break

    if not last:
        return "⚠️ No pending sends without outcomes for this customer."

    last["outcome"] = outcome

    # 1. Update per-customer threshold
    new_delta = update_threshold(customer_id, outcome)

    # 2. Online ML update
    ml_update(last["row"], outcome)

    # 3. Log it
    feedback_log.append({
        "customer_id": customer_id,
        "outcome": outcome,
        "rule_score": last["rule_score"],
        "final_score": last["final_score"],
        "reward": OUTCOME_REWARD[outcome],
        "new_threshold_delta": new_delta,
        "timestamp": datetime.now().strftime("%H:%M:%S"),
    })

    reward   = OUTCOME_REWARD[outcome]
    new_thr  = 50 + new_delta
    direction = "📈 Raised" if reward < 0 else "📉 Lowered"
    emoji    = {"converted":"🎉","clicked":"✅","ignored":"😐","opted_out":"🚨"}[outcome]

    return f"""
╔══════════════════════════════════════════════╗
  🔄 FEEDBACK RECEIVED — Model Updated
╚══════════════════════════════════════════════╝

  👤 Customer  : {customer_id}
  {emoji} Outcome   : {outcome.upper()}
  💰 Reward    : {reward:+.1f}

  🧠 WHAT THE MODEL LEARNED
  {direction} threshold for {customer_id}
  New personal threshold : {new_thr}/100
  (Was 50, now {50+new_delta})

  🤖 ML model updated with this event
  Total feedback events  : {len(feedback_log)}

  💡 INSIGHT
  {"Great signal! We'll be more willing to send to this customer." if reward > 0
   else "Noted. We'll be more selective next time — raising the bar." if reward == -0.3
   else "⚠️ Opt-out detected! Threshold raised significantly. Handle with care."}

  ⏱  {datetime.now().strftime("%H:%M:%S")}
"""


def show_learning_log():
    if not feedback_log:
        return "No feedback received yet. Score a customer and record an outcome!"

    lines = ["╔══════════════════════════════════════════════╗",
             "  🧠 LEARNING LOG — All Feedback Events",
             "╚══════════════════════════════════════════════╝\n"]

    for f in feedback_log:
        lines.append(f"  {f['timestamp']} | {f['customer_id']} | "
                     f"{f['outcome'].upper()} | reward:{f['reward']:+.1f} | "
                     f"score:{f['final_score']} | "
                     f"new threshold: {50+f['new_threshold_delta']:.0f}")

    lines.append(f"\n  Total Events : {len(feedback_log)}")
    lines.append(f"  Customers Tracked : {len(customer_history)}")
    lines.append(f"  ML Model Trained  : {'Yes ✅' if ml_trained else 'No — need first feedback'}")

    if customer_threshold_delta:
        lines.append("\n  📊 PER-CUSTOMER LEARNED THRESHOLDS")
        for cid, delta in customer_threshold_delta.items():
            bar = "▲" if delta > 0 else "▼" if delta < 0 else "—"
            lines.append(f"     {cid}: base 50 → {50+delta:.0f}  {bar}")

    return "\n".join(lines)



with gr.Blocks(theme=gr.themes.Base(primary_hue="blue", neutral_hue="slate"),
               title="Receptivity AI") as demo:

    gr.Markdown("""
    # ⚡ Receptivity AI — Self-Learning Edition
    ### Rule Engine + Online ML + Per-Customer Adaptive Thresholds
    *Scores signals → sends → learns from engagement → gets smarter every time*
    ---
    """)

    with gr.Tabs():

        # ── TAB 1: Score + Decide ─────────────────────────────
        with gr.TabItem("🎯 Score & Decide"):
            gr.Markdown("### Step 1 — Enter customer signals and get a hybrid decision")

            customer_id_input = gr.Textbox(
                label="Customer ID", placeholder="e.g. CUST_001",
                value="CUST_001"
            )

            with gr.Row():
                with gr.Column():
                    gr.Markdown("**⏰ Time & Device**")
                    hour       = gr.Slider(0, 23,   value=9,    step=1,   label="Hour of Day")
                    brightness = gr.Slider(0, 100,  value=75,   step=1,   label="Screen Brightness")
                    on_wifi    = gr.Checkbox(value=True,  label="On WiFi")
                    dnd        = gr.Checkbox(value=False, label="Do Not Disturb")
                    face_down  = gr.Checkbox(value=False, label="Phone Face Down")

                with gr.Column():
                    gr.Markdown("**📱 Friction & Engagement**")
                    recent_err = gr.Checkbox(value=False, label="Recent Error")
                    closed_app = gr.Checkbox(value=False, label="Recently Closed App")
                    swipe      = gr.Radio(["slow","medium","fast"], value="medium", label="Swipe Speed")
                    depth      = gr.Radio(["shallow","mid","deep"], value="shallow", label="Session Depth")

                with gr.Column():
                    gr.Markdown("**🧠 Emotion & Timing**")
                    sentiment  = gr.Slider(-1.0, 1.0, value=0.3, step=0.05, label="Sentiment Score")
                    dwell      = gr.Slider(2, 300,  value=60,   step=1,   label="Time on Screen (sec)")
                    response_t = gr.Slider(1, 30,   value=10,   step=1,   label="Avg Response Time (sec)")

            score_btn = gr.Button("⚡ Score & Decide", variant="primary", size="lg")
            score_out = gr.Textbox(label="Decision Output", lines=22)

            score_btn.click(
                fn=score_and_decide,
                inputs=[customer_id_input, hour, brightness, sentiment,
                        dnd, face_down, on_wifi, recent_err, closed_app,
                        swipe, depth, dwell, response_t],
                outputs=score_out
            )

        # ── TAB 2: Record Outcome ─────────────────────────────
        with gr.TabItem("🔄 Record Engagement"):
            gr.Markdown("""
            ### Step 2 — After sending, log what the customer actually did
            *This is where the AI learns. Every outcome updates the model instantly.*
            """)

            with gr.Row():
                cid_feedback = gr.Textbox(
                    label="Customer ID", placeholder="e.g. CUST_001",
                    value="CUST_001"
                )
                outcome_input = gr.Radio(
                    ["converted", "clicked", "ignored", "opted_out"],
                    label="What did the customer do?",
                    value="clicked"
                )

            gr.Markdown("""
            | Outcome | Meaning | Reward |
            |---|---|---|
            | 🎉 converted | Clicked + purchased | +1.0 |
            | ✅ clicked | Opened notification | +0.7 |
            | 😐 ignored | Delivered, not opened | -0.3 |
            | 🚨 opted_out | Unsubscribed | -1.0 |
            """)

            feedback_btn = gr.Button("📥 Record Outcome & Update Model",
                                     variant="primary", size="lg")
            feedback_out = gr.Textbox(label="Learning Output", lines=18)

            feedback_btn.click(
                fn=record_outcome,
                inputs=[cid_feedback, outcome_input],
                outputs=feedback_out
            )

        # ── TAB 3: Learning Log ───────────────────────────────
        with gr.TabItem("🧠 Learning Log"):
            gr.Markdown("""
            ### Full history of what the model has learned
            *Watch thresholds shift per customer as feedback accumulates*
            """)
            log_btn = gr.Button("🔍 Show Learning Log", variant="primary")
            log_out = gr.Textbox(label="Learning Log", lines=30)
            log_btn.click(fn=show_learning_log, inputs=[], outputs=log_out)

demo.launch()

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ba6dfa27e28ccfda9c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
